In [ ]:
# Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split    
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from math import pi
import sqlite3
import tqdm
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, LSTM,GlobalAveragePooling1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
# repeat vector
from tensorflow.keras.layers import RepeatVector, TimeDistributed, Dropout,Input
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [ ]:
#Load a single table from the sqlite database
# Create a connection to the database
conn = sqlite3.connect('../data/mill.db')
# Load the table into a pandas DataFrame
table_name = 'settings'
settings_DF = pd.read_sql_query("SELECT * from " + table_name, conn)

#load the signals tables
signal_DFs=[]
signal_names=['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
for signal_name in signal_names:
    signal_DF = pd.read_sql_query("SELECT * from " + signal_name, conn)
    signal_DFs.append(signal_DF)

#merge the signals into a single dataframe, as they all have the same length
merged_DF = pd.concat(signal_DFs, axis=1)
# remove duplicate ID column
merged_DF=merged_DF.loc[:,~merged_DF.columns.duplicated()]
merged_DF.head(10)

# convert cut_no to int
# Extract the numeric part of the cut_no column
merged_DF['cut_no'] = merged_DF['cut_no'].str.extract('(\d+)').astype(int)
# Repeat the conversion for the settings_DF
settings_DF['cut_no'] = settings_DF['cut_no'].str.extract('(\d+)').astype(int)

In [ ]:
merged_DF.head(10)

In [ ]:
settings_DF

In [ ]:
# Based on Figure 6 in the DataSheet, we can set a wear treshold to 0.25
wear_threshold = 0.25
# Create a binary classification column based on the wear threshold
settings_DF['wear_class'] = np.where(settings_DF['VB'] > wear_threshold, 1, 0)
# 1 for wear, 0 for no wear

# Merge the settings_DF with the merged_DF on cut_no
binary_class_DF = pd.merge(merged_DF, settings_DF[['cut_no', 'wear_class']], on='cut_no', how='left')
# Quick peek at the binary classification DataFrame
binary_class_DF.head(10)

In [ ]:
# Reformat the data into sequences
# Group the data by 'cut_no' to represent each time series
sequence_data = []
labels = []

for cut_no, group in binary_class_DF.groupby('cut_no'):
    # Drop non-time-series columns and convert to numpy array
    time_series = group.drop(columns=['cut_no', 'wear_class']).to_numpy()
    sequence_data.append(time_series)
    labels.append(group['wear_class'].iloc[0])  # Use the first label for the sequence

# Convert to numpy arrays
sequence_data = np.array(sequence_data, dtype=object)  # Object dtype for variable-length sequences
labels = np.array(labels)

print(f"Number of sequences: {len(sequence_data)}")
print(f"Shape of first sequence: {sequence_data[0].shape}")

In [ ]:
# Check if sequences have the same length
sequence_lengths = [len(seq) for seq in sequence_data]
unique_lengths = set(sequence_lengths)

print(f"Unique sequence lengths: {unique_lengths}")
if len(unique_lengths) == 1:
    print("All sequences have the same length.")
else:
    print("Sequences have variable lengths.")

# Identify and print sequences with lengths different from the most common length
from collections import Counter

# Count occurrences of each sequence length
length_counts = Counter(sequence_lengths)
most_common_length = length_counts.most_common(1)[0][0]

print(f"Most common sequence length: {most_common_length}")
print("Sequences with different lengths:")
for i, length in enumerate(sequence_lengths):
    if length != most_common_length:
        print(f"Sequence index {i} has length {length}")

In [ ]:
# Drop sequences with lengths different from the most common length
filtered_sequences = []
filtered_labels = []
for seq, label in zip(sequence_data, labels):
    if len(seq) == most_common_length:
        filtered_sequences.append(seq)
        filtered_labels.append(label)   

In [ ]:
# Convert filtered_labels to categorical format for multi-class classification
filtered_labels_categorical = to_categorical(filtered_labels)

# Convert filtered_sequences to numpy array for compatibility
filtered_sequences = np.array(filtered_sequences, dtype=object)

# Split the filtered data into training and testing sets for an autoencoder, wicht means we don't need the labels but the train data contains only sequences with "positive" wear
# Split the data into two categories based on the label
wear_class_0 = [seq for seq, label in zip(filtered_sequences, filtered_labels) if label == 0]
wear_class_1 = [seq for seq, label in zip(filtered_sequences, filtered_labels) if label == 1]

# Use 90% of wear class "0" for training and the remaining 10% for testing
train_size = int(0.8 * len(wear_class_0))
sequence_train = wear_class_0[:train_size]
sequence_test = wear_class_0[train_size:] + wear_class_1


# Convert sequences to 3D shape for LSTM input
sequence_train = np.array([seq for seq in sequence_train])
sequence_test = np.array([seq for seq in sequence_test])
# Ensure the sequences are 3D (samples, timesteps, features)
sequence_train = np.array([seq.reshape(-1, seq.shape[1]) for seq in sequence_train])
sequence_test = np.array([seq.reshape(-1, seq.shape[1]) for seq in sequence_test])


In [ ]:
# Convert sequence_train and sequence_test to proper numpy arrays with float dtype
X_train = np.array(sequence_train, dtype=np.float32)
X_test = np.array(sequence_test, dtype=np.float32)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Flatten the 3D array to 2D for scaling
num_samples, num_timesteps, num_features = X_train.shape
X_train_reshaped = X_train.reshape(-1, num_features)  # Shape: (num_samples * num_timesteps, num_features)

# LSTM tends to be sensitive to the length of sequences, so we might want to downsample the sequences to reduce the number of timesteps.
# Downsample the sequences (e.g., keep every 5th timestep)
X_train = X_train[:, ::5, :]
X_test = X_test[:, ::5, :]
print(f"New shape of X_train: {X_train.shape}")

#Clip the values to a range to avoid extreme values
X_train = np.clip(X_train, -10, 10)
X_test = np.clip(X_test, -10, 10)


# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reshaped)

# Reshape back to 3D
X_train = X_train_scaled.reshape(num_samples, num_timesteps, num_features)


# Repeat for X_test
num_samples_test, num_timesteps_test, num_features_test = X_test.shape
X_test_reshaped = X_test.reshape(-1, num_features_test)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(num_samples_test, num_timesteps_test, num_features_test)

# Verify the new mean and std
print(f"New Mean: {np.mean(X_train, axis=(0, 1))}")
print(f"New Std: {np.std(X_train, axis=(0, 1))}")

In [ ]:
print(f"X_train mean: {np.mean(X_train, axis=(0, 1))}")
print(f"X_train std: {np.std(X_train, axis=(0, 1))}")
print(f"X_train max: {np.max(X_train)}")
print(f"X_train min: {np.min(X_train)}")
print(f"X_train variance: {np.var(X_train, axis=(0, 1))}")

In [ ]:
# Interactive visualization of sequences
@interact(index=(0, len(sequence_test) - 1))
def visualize_sequence(index=0):
    # Extract the sequence at the given index
    # clip the sequence so only every 10th timestep is shown
    sequence = sequence_test[index]
    sequence = sequence[::5, :]

    # Plot each feature in the sequence
    plt.figure(figsize=(12, 8))
    for feature_idx in range(sequence.shape[1]):
        plt.plot(sequence[:, feature_idx], label=f'Feature {feature_idx + 1}')

    plt.title(f'Sequence {index} Visualization')
    plt.xlabel('Timesteps')
    plt.ylabel('Feature Values')
    plt.legend()
    plt.show()

In [ ]:
# Define the AutoEncoder model
input_dim = X_train.shape[2]  # Number of features in each timestep
timesteps = X_train.shape[1]  # Update to the new sequence length (e.g., 900)
autoencoder = Sequential([
    LSTM(64, activation='tanh', return_sequences=True, input_shape=(timesteps, input_dim)),
    Dropout(0.2),
    LSTM(32, activation='tanh', return_sequences=False),

    Dense(16, activation='relu'), #Latent space representation
    RepeatVector(timesteps),    #Latent space to sequence

    LSTM(32, activation='tanh', return_sequences=True),
    Dropout(0.2),
    LSTM(64, activation='tanh', return_sequences=True),
    Dense(input_dim, activation='linear')
])

In [ ]:
# Compile the AutoEncoder
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

# reduce the learning rate to 0.0001
#autoencoder.compile(optimizer=Adam(learning_rate=0.0001, clipnorm=1.0), loss='mse')

# Display the model summary
autoencoder.summary()

In [ ]:
# add a learning rate scheduler to deal with potential overfitting
# Reduce learning rate when validation loss plateaus
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

In [ ]:
# Train the AutoEncoder
history = autoencoder.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    callbacks=[lr_scheduler],
    verbose=2  # Print epoch-level loss only
)

In [ ]:
# plit the training history
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Evaluate the reconstruction error
reconstruction = autoencoder.predict(X_test)
reconstruction_error = np.mean(np.square(X_test - reconstruction), axis=(1, 2))

# Plot reconstruction error
plt.hist(reconstruction_error, bins=50)
plt.xlabel('Reconstruction Error')
plt.ylabel('Frequency')
plt.title('Reconstruction Error Distribution')
plt.show()

# Optionally, set a threshold for anomaly detection based on reconstruction error
threshold = np.percentile(reconstruction_error, 95)  # Example: 95th percentile
print(f"Anomaly detection threshold: {threshold}")